In [3]:
from pathlib import Path
import os

def _find_package_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "PACKAGE_ROOT.marker").is_file():
            return candidate
    raise RuntimeError(
        "Cannot locate PACKAGE_ROOT.marker. "
        "Run this notebook from inside the extracted RFRR submission package."
    )

PACKAGE_ROOT = _find_package_root()


ART = Path(
    os.environ.get("RFRR_ARTIFACTS_DIR", PACKAGE_ROOT / "data" )
).expanduser().resolve()

OUTPUT_ROOT = Path(
    os.environ.get("RFRR_OUTPUT_ROOT", PACKAGE_ROOT / "outputs")
).expanduser().resolve()

MODE = os.environ.get("RFRR_MODE", "intervals")  # intervals / all
OUT = OUTPUT_ROOT / MODE

HISTORICAL_RESULTS = Path(
    os.environ.get(
        "RFRR_HISTORICAL_RESULTS",
        PACKAGE_ROOT / "data" / "historical_results"
    )
).expanduser().resolve()

HISTORICAL_ZIP = Path(
    os.environ.get(
        "RFRR_HISTORICAL_ZIP",
        PACKAGE_ROOT / "data" / "01_alignment_checks.zip"
    )
).expanduser().resolve()

OURS_CSV = ART / "ours_aligned_signals.csv"
IAPL_CSV = ART / "iapl_aligned_predictions_seven_domains.csv"

SPLIT_SEED = 20261118  
TRAIN_SEEDS = (20261118, 20261119, 20261120)
BOOTSTRAP_SEED = 20261121
N_BOOTSTRAP = 2000
SAVE_EVERY = 100
BOOTSTRAP_TRAINED_RESULTS = True
EPOCHS = 70
PAIRS_PER_GROUP = 1024
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4
RESIDUAL_PENALTY = 1e-3
ROBUST_TAU = 0.20
HIDDEN1, HIDDEN2 = 16, 8
THRESHOLD, AI_PRIOR = 0.5, 0.10
NATURAL_FOLDS = 2
SEED = SPLIT_SEED
BUDGETS = (0.05, 0.10, 0.20)
TARGET_RECOVERY = 0.50
AURBC_MAX_BUDGET = 0.30
DOMAINS = ("sd_v1_5", "wukong", "glide", "vqdm", "adm", "biggan")
DISPLAY_NAME = dict(zip(DOMAINS, ("SD1.5", "Wukong", "GLIDE", "VQDM", "ADM", "BigGAN")))
FEATURE_COLS = ["u_iapl", "u_primary", "u_frequency", "u_residual",
                "gap_frequency_iapl", "gap_residual_iapl"]
METRICS = ("recovery_at_10", "aurbc_0_30", "required_budget_at_50")
VARIANTS = {
    "DirectPair": dict(structure="direct", beta=2.0, robust=False),
    "ResidualNoGate": dict(structure="residual", beta=2.0, robust=False),
    "RFRR-Avg": dict(structure="gated", beta=2.0, robust=False),
    "RFRR-Beta0": dict(structure="gated", beta=0.0, robust=False),
    "RFRR-WG": dict(structure="gated", beta=2.0, robust=True),
}


In [4]:
import hashlib
import json
import platform
import random
import warnings
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from tqdm.auto import tqdm
except ImportError:
    warnings.warn("安装 tqdm 可显示完整进度条。当前使用简要进度输出。")
    def tqdm(iterable, desc="", **kwargs):
        total = len(iterable) if hasattr(iterable, "__len__") else None
        for i, item in enumerate(iterable):
            if i == 0 or (i + 1) % 100 == 0:
                print(desc, i + 1, "/", total)
            yield item


def require(ok, message):
    if not bool(ok):
        raise ValueError(message)


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(2**20), b""):
            h.update(chunk)
    return h.hexdigest()


def save_csv(name, frame):
    path = OUT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8-sig")
    temp.replace(path)


def save_json(name, obj):
    path = OUT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)


def atomic_npz(path, **arrays):
    path = Path(path)
    with path.with_suffix(".tmp").open("wb") as stream:
        np.savez_compressed(stream, **arrays)
    path.with_suffix(".tmp").replace(path)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def historical_csv(name):
    path = HISTORICAL_RESULTS / name
    if path.is_file():
        return pd.read_csv(path, dtype={"physical_id": str, "sample_id": str})
    if HISTORICAL_ZIP.is_file():
        with zipfile.ZipFile(HISTORICAL_ZIP) as archive:
            return pd.read_csv(archive.open(name), dtype={"physical_id": str, "sample_id": str})
    raise FileNotFoundError(f"找不到 {path} 或 {HISTORICAL_ZIP}")


require(MODE in ("intervals", "all"), "MODE 须为 intervals 或 all。")
require(len(TRAIN_SEEDS) == 3 and len(set(TRAIN_SEEDS)) == 3, "请设置3个不同的训练种子。")
require(N_BOOTSTRAP > 0 and SAVE_EVERY > 0, "Bootstrap 配置必须为正整数。")
OUT.mkdir(parents=True, exist_ok=True)
if MODE == "all":
    try:
        import torch
        from torch import nn
        import torch.nn.functional as F
    except ImportError as exc:
        raise ImportError("训练需要 PyTorch。请使用原RFRR的CUDA内核。") from exc
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.use_deterministic_algorithms(True, warn_only=True)
    print("训练设备", DEVICE)


## 1 · 复用原始对齐、物理自然图两折及池内秩


In [5]:
require(OURS_CSV.is_file(), f"缺少 {OURS_CSV}")
require(IAPL_CSV.is_file(), f"缺少 {IAPL_CSV}")

ours = pd.read_csv(OURS_CSV, encoding="utf-8-sig")
iapl = pd.read_csv(IAPL_CSV, encoding="utf-8-sig")

need_ours = {
    "sample_id", "domain_id", "true_label", "class_index", "physical_id",
    "score_ai", "score_semantic", "score_frequency", "score_residual"
}
need_iapl = {"sample_id", "domain_id", "true_label", "score_ai"}

require(need_ours <= set(ours.columns),
        f"28号缺列：{sorted(need_ours-set(ours.columns))}")
require(need_iapl <= set(iapl.columns),
        f"IAPL表缺列：{sorted(need_iapl-set(iapl.columns))}")

ours = ours.loc[ours.domain_id.isin(DOMAINS)].copy()
iapl = iapl.loc[iapl.domain_id.isin(DOMAINS)].copy()

require(len(ours) == 72000 and ours.sample_id.is_unique,
        "28号六域应为72,000行且sample_id唯一。")
require(len(iapl) == 72000 and iapl.sample_id.is_unique,
        "IAPL六域应为72,000行且sample_id唯一。")

frame = ours.merge(
    iapl[["sample_id","domain_id","true_label","score_ai"]]
        .rename(columns={"score_ai":"iapl_score"}),
    on=["sample_id","domain_id","true_label"],
    how="inner", validate="one_to_one"
)

require(len(frame) == 72000, "IAPL与三分支未完全一一对齐。")
require(frame["true_label"].isin([0,1]).all(), "标签异常。")
for c in ["score_ai","score_semantic","score_frequency","score_residual","iapl_score"]:
    require(frame[c].between(0,1).all() and np.isfinite(frame[c]).all(),
            f"{c} 非有限或不在[0,1]。")

# 六域共享自然图按物理ID核验
nat = frame.loc[frame.true_label.eq(0)].copy()
require(nat.physical_id.nunique() == 6000,
        f"自然物理图应为6000张，实际 {nat.physical_id.nunique():,}")
require(nat.groupby("physical_id").domain_id.nunique().eq(6).all(),
        "自然物理图没有完整覆盖六域。")

# 每个检测分数对于同一自然图跨域应一致
for c in ["score_ai","score_semantic","score_frequency","score_residual","iapl_score","class_index"]:
    require(nat.groupby("physical_id")[c].nunique().eq(1).all(),
            f"共享自然图跨域 {c} 不一致。")

# AI图物理哈希不能跨生成器重复
ai = frame.loc[frame.true_label.eq(1)]
require(ai.groupby("physical_id").domain_id.nunique().le(1).all(),
        "AI图存在跨生成器物理重复，LOGO可能泄漏。")

alignment = pd.DataFrame([
    {"check":"six_domain_rows", "value":len(frame), "passed":len(frame)==72000},
    {"check":"shared_natural_physical_images", "value":nat.physical_id.nunique(), "passed":nat.physical_id.nunique()==6000},
    {"check":"ai_cross_generator_duplicate_groups",
     "value":int((ai.groupby("physical_id").domain_id.nunique()>1).sum()),
     "passed":bool(ai.groupby("physical_id").domain_id.nunique().le(1).all())},
])
save_csv("01_alignment_checks.csv", alignment)
display(alignment)


,check,value,passed
0,six_domain_rows,72000,True
1,shared_natural_physical_images,6000,True
2,ai_cross_generator_duplicate_groups,0,True


In [6]:
natural_base = (
    nat.sort_values(["physical_id","domain_id"])
       .drop_duplicates("physical_id")
       .copy()
)
require(len(natural_base) == 6000, "canonical natural 应为6000。")

hashes = np.asarray(sorted(natural_base.physical_id.astype(str).unique()))
rng = np.random.default_rng(SEED)
perm = rng.permutation(hashes)
parts = np.array_split(perm, NATURAL_FOLDS)
NAT_PARTS = {i:set(map(str, p)) for i,p in enumerate(parts)}

split_rows = []
for test_nat_fold in range(NATURAL_FOLDS):
    test_ids = NAT_PARTS[test_nat_fold]
    train_ids = set(hashes) - test_ids
    require(train_ids.isdisjoint(test_ids), "自然图训练/测试物理ID交叉。")
    split_rows.append({
        "test_natural_fold": test_nat_fold,
        "train_natural_n": len(train_ids),
        "test_natural_n": len(test_ids),
        "physical_overlap": len(train_ids & test_ids),
    })

split_check = pd.DataFrame(split_rows)
save_csv("01_natural_split_checks.csv", split_check)
display(split_check)


,test_natural_fold,train_natural_n,test_natural_n,physical_overlap
0,0,3000,3000,0
1,1,3000,3000,0


In [7]:
def empirical_rank01(values):
    s = pd.Series(np.asarray(values, float))
    # 高分 -> 高percentile；average处理并列
    u = s.rank(method="average", pct=True).to_numpy(float)
    eps = 0.5 / max(2, len(u))
    return np.clip(u, eps, 1-eps)

def make_pool(generator, natural_ids):
    # held/source generator AI
    ai_part = frame.loc[
        frame.domain_id.eq(generator) & frame.true_label.eq(1)
    ].copy()

    # 共享自然图使用canonical物理副本
    nat_part = natural_base.loc[
        natural_base.physical_id.astype(str).isin(set(map(str,natural_ids)))
    ].copy()
    nat_part["domain_id"] = "shared_natural"
    nat_part["sample_id"] = "natural::" + nat_part.physical_id.astype(str)

    full = pd.concat([ai_part, nat_part], ignore_index=True)

    # 检测前先验权重
    is_ai = full.true_label.eq(1).to_numpy()
    w = np.empty(len(full), float)
    w[is_ai] = AI_PRIOR / is_ai.sum()
    w[~is_ai] = (1-AI_PRIOR) / (~is_ai).sum()
    full["input_weight"] = w

    release = full.loc[full.score_ai.lt(THRESHOLD)].copy().reset_index(drop=True)
    require(release.true_label.nunique()==2,
            f"{generator} 当前自然子集放行池缺少某一类。")

    release["u_iapl"] = empirical_rank01(release.iapl_score)
    release["u_primary"] = empirical_rank01(release.score_ai)
    release["u_frequency"] = empirical_rank01(release.score_frequency)
    release["u_residual"] = empirical_rank01(release.score_residual)
    release["gap_frequency_iapl"] = np.abs(release.u_frequency-release.u_iapl)
    release["gap_residual_iapl"] = np.abs(release.u_residual-release.u_iapl)

    require(np.isfinite(release[FEATURE_COLS].to_numpy(float)).all(),
            "RFRR秩特征出现非有限值。")
    return full, release

def logit_np(u):
    u = np.clip(np.asarray(u,float),1e-6,1-1e-6)
    return np.log(u/(1-u))


In [8]:
require(not frame[["physical_id", "class_index", "sample_id"]].isna().any().any(), "图像ID/类别缺失。")
require(frame.groupby("physical_id").class_index.nunique().eq(1).all(), "同一物理图的类别不一致。")
require(frame.groupby("physical_id").true_label.nunique().eq(1).all(), "同一物理图标签冲突。")
for generator in DOMAINS:
    full_g = frame.loc[frame.domain_id.eq(generator)]
    counts = full_g.true_label.value_counts()
    require(counts.to_dict() == {0: 6000, 1: 6000} or counts.to_dict() == {1: 6000, 0: 6000},
            f"{generator} 需要原始6000张AI＋6000张自然图。")
    require(full_g.physical_id.is_unique, f"{generator} 内存在精确重复图片。")

physical_class = frame.drop_duplicates("physical_id").set_index("physical_id").class_index
classes = pd.Index(sorted(frame.class_index.unique()))
require(len(classes) == 1000, "原GenImage清单应覆盖1000个ImageNet类别。")

historical = historical_csv("02_test_ranking_scores.csv")
historical["seed"] = -1  # 标识历史单次开发运行
required_columns = {"heldout_generator", "test_natural_fold", "method", "sample_id",
                    "physical_id", "true_label", "ranking_score", "input_weight"}
require(required_columns <= set(historical), "历史预测字段缺失。")
require(set(historical.heldout_generator) == set(DOMAINS), "历史预测未覆盖六域。")
require(set(historical.method) >= {"IAPL", "RFRR-Avg"}, "历史预测缺主对照。")

# 完整记录由make_pool构造，可核查每折候选池与完整清单类别权重。
full_records = {}
for generator in DOMAINS:
    for fold in range(NATURAL_FOLDS):
        full, released = make_pool(generator, NAT_PARTS[fold])
        full_records[(generator, fold)] = full
        trial = historical.loc[historical.heldout_generator.eq(generator)
                               & historical.test_natural_fold.eq(fold)]
        expected = released.set_index("physical_id")[["true_label", "input_weight"]].sort_index()
        for method, part in trial.groupby("method"):
            require(part.physical_id.is_unique, "历史预测同一试验同一方法内物理ID重复。")
            actual = part.set_index("physical_id")[["true_label", "input_weight"]].sort_index()
            pd.testing.assert_frame_equal(actual, expected, check_dtype=False, atol=1e-12, rtol=1e-10)
require(historical.ranking_score.notna().all()
        and np.isfinite(historical.ranking_score.to_numpy()).all(), "历史分数无效。")
print("历史预测与原始放行池、类别权重已逐图对齐。")
display(historical_csv("03_macro_main_table.csv"))

# 参数和输入哈希约束续跑，配置改变时提示改输出目录。
contract = dict(code_sha256='109f65d175ab1a5e4322112ac21a783fbfc8999cdf967c20456bff193a4806a6', version="rfrr-closeout-v1", mode=MODE, split_seed=SPLIT_SEED,
                train_seeds=list(TRAIN_SEEDS), bootstrap_seed=BOOTSTRAP_SEED,
                bootstrap_repetitions=N_BOOTSTRAP, epochs=EPOCHS,
                runtime=dict(python=platform.python_version(), numpy=np.__version__,
                             pandas=pd.__version__,
                             torch=torch.__version__ if MODE == "all" else None),
                pairs_per_group=PAIRS_PER_GROUP, learning_rate=LEARNING_RATE,
                weight_decay=WEIGHT_DECAY, residual_penalty=RESIDUAL_PENALTY,
                robust_tau=ROBUST_TAU, hidden=[HIDDEN1, HIDDEN2], variants=VARIANTS,
                threshold=THRESHOLD, ai_prior=AI_PRIOR,
                target_recovery=TARGET_RECOVERY, aurbc_max_budget=AURBC_MAX_BUDGET,
                features=FEATURE_COLS, source_sha=sha256_file(OURS_CSV),
                iapl_sha=sha256_file(IAPL_CSV), rank_features="unweighted batch percentiles",
                uncertainty="conditional on predictions; common ImageNet class resampling")
old_contract = OUT / "protocol.json"
if old_contract.is_file():
    require(json.loads(old_contract.read_text(encoding="utf-8")) == contract,
            "当前输出目录已有不同输入/参数。请为本次设置使用新的OUT目录。")
save_json("protocol.json", contract)


历史预测与原始放行池、类别权重已逐图对齐。


,method,recovery_at_5,recovery_at_10,recovery_at_20,review_precision_at_10,remaining_fnr_at_10,required_budget_at_50,aurbc_0_30,worst_generator_recovery_at_10,best_generator_recovery_at_10
0,IAPL,0.491200,0.722660,0.871185,0.421957,0.209994,0.059812,0.721674,0.447225,0.882353
1,LR-Fusion,0.394128,0.645506,0.868300,0.438434,0.193567,0.076973,0.683439,0.440541,0.816343
2,RFRR-Avg,0.494461,0.736217,0.886355,0.433787,0.198668,0.059022,0.732264,0.449063,0.882353
3,RFRR,0.493281,0.727915,0.876359,0.428241,0.203917,0.058592,0.725544,0.459351,0.877451


## 2 · 同容量的五种训练版本


In [9]:
if MODE == 'all':
    class ResidualRankNet(nn.Module):
        def __init__(self, d):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(d, HIDDEN1),
                nn.ReLU(),
                nn.Linear(HIDDEN1, HIDDEN2),
                nn.ReLU(),
                nn.Linear(HIDDEN2, 1),
            )

        def forward(self, x):
            return self.net(x).squeeze(-1)


In [10]:
def _score_groups(scores, weights, positives):
    order = np.argsort(-np.asarray(scores,float), kind="mergesort")
    s = np.asarray(scores,float)[order]
    w = np.asarray(weights,float)[order]
    p = np.asarray(positives,bool)[order]

    starts = np.r_[0, np.flatnonzero(np.diff(s)!=0)+1]
    group_w = np.add.reduceat(w, starts)
    group_pos = np.add.reduceat(w*p, starts)
    return group_w, group_pos

def recovery_at_budget(release, scores, budget):
    weights = release.input_weight.to_numpy(float)
    pos = release.true_label.eq(1).to_numpy()
    total_fn = float(weights[pos].sum())
    total_pool = float(weights.sum())
    require(total_fn>0 and total_pool>0, "评价池质量异常。")

    gw, gp = _score_groups(scores, weights, pos)
    amount = budget*total_pool
    before = np.cumsum(gw)-gw
    frac = np.divide(
        np.clip(amount-before,0,gw), gw,
        out=np.zeros_like(gw), where=gw>0
    )
    recovered = float(np.sum(frac*gp))
    review_mass = min(amount,total_pool)
    return {
        "recovery": recovered/total_fn,
        "review_precision": recovered/review_mass if review_mass>0 else np.nan,
        "remaining_fnr": (total_fn-recovered)/AI_PRIOR,
    }

def required_budget_for_target(release, scores, target):
    weights = release.input_weight.to_numpy(float)
    pos = release.true_label.eq(1).to_numpy()
    total_fn = float(weights[pos].sum())
    total_pool = float(weights.sum())
    need = target*total_fn

    gw, gp = _score_groups(scores, weights, pos)
    cum_pos = np.cumsum(gp)
    k = int(np.searchsorted(cum_pos, need, side="left"))
    require(k < len(gp), "无法达到目标追回率。")

    pos_before = float(cum_pos[k-1]) if k>0 else 0.0
    mass_before = float(gw[:k].sum())
    require(gp[k] > 0, "目标交叉分数组没有正类质量。")
    frac = float(np.clip((need-pos_before)/gp[k],0,1))
    budget_mass = mass_before + frac*gw[k]
    return budget_mass/total_pool

def aurbc(release, scores, max_budget=AURBC_MAX_BUDGET):
    grid = np.linspace(0,max_budget,31)
    rec = np.array([recovery_at_budget(release,scores,b)["recovery"] for b in grid])
    return float(np.trapz(rec,grid)/max_budget)

def evaluate_method(release, scores):
    row = {}
    for b in BUDGETS:
        m = recovery_at_budget(release,scores,b)
        row[f"recovery_at_{int(100*b)}"] = m["recovery"]
        if abs(b-0.10)<1e-12:
            row["review_precision_at_10"] = m["review_precision"]
            row["remaining_fnr_at_10"] = m["remaining_fnr"]
    row["required_budget_at_50"] = required_budget_for_target(
        release,scores,TARGET_RECOVERY
    )
    row["aurbc_0_30"] = aurbc(release,scores)
    return row


In [11]:
def variant_score(model, x, structure):
    delta = model(x)
    if structure == "direct":
        return delta, delta
    u = x[:, 0].clamp(1e-6, 1 - 1e-6)
    base = torch.log(u / (1 - u))
    correction = (1 - u) * delta if structure == "gated" else delta
    return base + correction, delta


def matched_train(pools, variant, seed, description):
    config = VARIANTS[variant]
    set_seed(seed)
    model = ResidualRankNet(len(FEATURE_COLS)).to(DEVICE)
    initial_sha = hashlib.sha256(b"".join(
        x.detach().cpu().numpy().tobytes() for x in model.state_dict().values()
    )).hexdigest()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    rng = np.random.default_rng(seed)
    tensors = {}
    indices = {}
    for generator in sorted(pools):
        part = pools[generator]
        tensors[generator] = torch.tensor(part[FEATURE_COLS].to_numpy(np.float32), device=DEVICE)
        y = part.true_label.to_numpy()
        indices[generator] = (np.flatnonzero(y == 1), np.flatnonzero(y == 0))
        require(min(map(len, indices[generator])) > 0, "训练池缺少正负样本。")
    pair_digest = hashlib.sha256()
    history = []
    model.train()
    for epoch in tqdm(range(EPOCHS), desc=description, unit="轮", leave=False):
        group_losses, penalties = [], []
        for generator in sorted(tensors):
            pos, neg = indices[generator]
            ip = rng.choice(pos, size=PAIRS_PER_GROUP, replace=len(pos) < PAIRS_PER_GROUP)
            jn = rng.choice(neg, size=PAIRS_PER_GROUP, replace=len(neg) < PAIRS_PER_GROUP)
            pair_digest.update(ip.astype(np.int64).tobytes())
            pair_digest.update(jn.astype(np.int64).tobytes())
            xp, xn = tensors[generator][ip], tensors[generator][jn]
            sp, dp = variant_score(model, xp, config["structure"])
            sn, dn = variant_score(model, xn, config["structure"])
            hard_weight = 1 + config["beta"] * (1 - xp[:, 0])
            group_losses.append((hard_weight * F.softplus(-(sp - sn))).mean())
            penalties.append((dp.square().mean() + dn.square().mean()) / 2)
        losses = torch.stack(group_losses)
        main = (ROBUST_TAU * (torch.logsumexp(losses / ROBUST_TAU, dim=0)
                              - np.log(len(losses)))) if config["robust"] else losses.mean()
        objective = main + RESIDUAL_PENALTY * torch.stack(penalties).mean()
        require(torch.isfinite(objective).item(), "训练出现非有限损失。")
        optimizer.zero_grad()
        objective.backward()
        optimizer.step()
        history.append(dict(epoch=epoch + 1, loss=float(objective.detach().cpu())))
    return model, pd.DataFrame(history), dict(initial_sha256=initial_sha,
                                              pairs_sha256=pair_digest.hexdigest())


def run_models():
    chunks, metric_rows, match_rows = [], [], []
    tasks = [(g, f, seed, variant) for g in DOMAINS for f in range(NATURAL_FOLDS)
             for seed in TRAIN_SEEDS for variant in VARIANTS]
    pools_cache = {}
    for generator in DOMAINS:
        for fold in range(NATURAL_FOLDS):
            train_ids = set(hashes) - NAT_PARTS[fold]
            pools = {g: make_pool(g, train_ids)[1] for g in DOMAINS if g != generator}
            query = make_pool(generator, NAT_PARTS[fold])[1]
            train_physical = set().union(*(set(p.physical_id) for p in pools.values()))
            require(train_physical.isdisjoint(set(query.physical_id)),
                    "训练与测试物理图像交叉。")
            pools_cache[(generator, fold)] = (pools, query)
            baseline = historical.loc[historical.heldout_generator.eq(generator)
                                      & historical.test_natural_fold.eq(fold)
                                      & historical.method.eq("IAPL")].copy()
            for seed in TRAIN_SEEDS:
                part = baseline.copy()
                part["seed"] = seed
                chunks.append(part)
                metric_rows.append(dict(heldout_generator=generator, test_natural_fold=fold,
                                        seed=seed, method="IAPL",
                                        **evaluate_method(query, query.iapl_score.to_numpy())))

    for generator, fold, seed, variant in tqdm(tasks, desc="三种子×六域×两折×五版本", unit="模型"):
        # No generator/fold/variant offset: exact same init and pairs within a task family.
        tag = f"{generator}_fold{fold}_seed{seed}_{variant}"
        job = OUT / "jobs" / tag
        job.mkdir(parents=True, exist_ok=True)
        marker = job / "complete.json"
        pools, query = pools_cache[(generator, fold)]
        if marker.is_file():
            done = json.loads(marker.read_text(encoding="utf-8"))
            for name, expected_sha in done["files"].items():
                require(sha256_file(job / name) == expected_sha, f"{tag} 缓存校验失败 {name}")
            prediction = pd.read_csv(job / "predictions.csv.gz",
                                     dtype={"physical_id": str, "sample_id": str})
            meta = done["matching"]
            metric = done["metric"]
        else:
            model, history, meta = matched_train(pools, variant, seed, tag)
            model.eval()
            with torch.no_grad():
                x = torch.tensor(query[FEATURE_COLS].to_numpy(np.float32), device=DEVICE)
                score, _ = variant_score(model, x, VARIANTS[variant]["structure"])
                score = score.detach().cpu().numpy().astype(float)
            prediction = pd.DataFrame(dict(
                heldout_generator=generator, test_natural_fold=fold, seed=seed, method=variant,
                sample_id=query.sample_id.to_numpy(), physical_id=query.physical_id.to_numpy(),
                true_label=query.true_label.to_numpy(), ranking_score=score,
                input_weight=query.input_weight.to_numpy()))
            metric = evaluate_method(query, score)
            path = job / "predictions.csv.gz"
            temp = job / "predictions.tmp.gz"
            prediction.to_csv(temp, index=False, compression="gzip")
            temp.replace(path)
            history.to_csv(job / "history.csv", index=False)
            atomic_npz(job / "weights.npz", **{key: value.detach().cpu().numpy()
                                             for key, value in model.state_dict().items()})
            save_json(str((marker).relative_to(OUT)), dict(
                matching=meta, metric=metric,
                files={name: sha256_file(job / name)
                       for name in ("predictions.csv.gz", "history.csv", "weights.npz")}))
            del model
        require(len(prediction) == len(query)
                and set(prediction.physical_id) == set(query.physical_id), f"{tag} 查询池不一致。")
        chunks.append(prediction)
        metric_rows.append(dict(heldout_generator=generator, test_natural_fold=fold,
                                seed=seed, method=variant, **metric))
        match_rows.append(dict(heldout_generator=generator, test_natural_fold=fold,
                               seed=seed, method=variant, **meta))
    matches = pd.DataFrame(match_rows)
    key = ["heldout_generator", "test_natural_fold", "seed"]
    require(matches.groupby(key).initial_sha256.nunique().eq(1).all(), "模型初始化未匹配。")
    require(matches.groupby(key).pairs_sha256.nunique().eq(1).all(), "正负样本对序列未匹配。")
    save_csv("02_matching_checks.csv", matches)
    metrics = pd.DataFrame(metric_rows)
    save_csv("03_seed_trial_metrics.csv", metrics)
    summary = metrics.groupby(["seed", "method"], sort=False)[list(METRICS)].mean().reset_index()
    save_csv("03_seed_macro_metrics.csv", summary)
    stability = summary.groupby("method")[list(METRICS)].agg(["mean", "std", "min", "max"])
    stability.columns = ["_".join(col) for col in stability.columns]
    save_csv("03_seed_stability.csv", stability.reset_index())
    iapl = summary.loc[summary.method.eq("IAPL")].set_index("seed")[list(METRICS)]
    delta = summary.loc[~summary.method.eq("IAPL")].copy()
    for metric in METRICS:
        delta["delta_" + metric] = delta[metric] - delta.seed.map(iapl[metric])
    save_csv("03_seed_deltas_vs_iapl.csv", delta)
    display(summary)
    display(delta)
    return pd.concat(chunks, ignore_index=True)


trained = None
if MODE == "all":
    trained = run_models()
else:
    print("intervals 模式复用历史预测。切换 all 后运行三种子和机制消融。")


intervals 模式复用历史预测。切换 all 后运行三种子和机制消融。


## 3 · 复用排序、共同类别 Bootstrap


In [12]:
def prepare_plans(predictions):
    predictions = predictions.copy()
    predictions["class_index"] = predictions.physical_id.map(physical_class)
    require(predictions.class_index.notna().all(), "部分预测找不到ImageNet类别。")
    plans = []
    for keys, part in predictions.groupby(
        ["heldout_generator", "test_natural_fold", "seed", "method"], sort=False
    ):
        generator, fold, seed, method = keys
        require(part.physical_id.is_unique, "同模型同试验重复预测。")
        require(np.isfinite(part.ranking_score).all(), "存在无效分数。")
        full = full_records[(generator, fold)]
        y_all = full.true_label.to_numpy()
        cls_all = classes.get_indexer(full.class_index)
        pre_counts = np.stack([np.bincount(cls_all[y_all == label], minlength=len(classes))
                               for label in (0, 1)])
        part = part.sort_values("ranking_score", ascending=False, kind="stable")
        s = part.ranking_score.to_numpy()
        starts = np.r_[0, np.flatnonzero(s[1:] != s[:-1]) + 1]
        plans.append(dict(key=keys, y=part.true_label.to_numpy(dtype=int),
                          classes=classes.get_indexer(part.class_index),
                          starts=starts, pre_counts=pre_counts))
    return plans


def plan_metrics(plan, multiplicities):
    n0, n1 = plan["pre_counts"] @ multiplicities
    if n0 <= 0 or n1 <= 0:
        return np.full(3, np.nan)
    one_weight = np.array([(1 - AI_PRIOR) / n0, AI_PRIOR / n1])
    y = plan["y"]
    weights = one_weight[y] * multiplicities[plan["classes"]]
    gw = np.add.reduceat(weights, plan["starts"])
    gp = np.add.reduceat(weights * y, plan["starts"])
    active = gw > 0
    gw, gp = gw[active], gp[active]
    total, positives = gw.sum(), gp.sum()
    if total <= 0 or positives <= 0:
        return np.full(3, np.nan)
    cw, cp = np.cumsum(gw), np.cumsum(gp)
    xp = np.r_[0, cw / total]
    yp = np.r_[0, cp / positives]
    rec10 = np.interp(0.1, xp, yp)
    grid = np.linspace(0, AURBC_MAX_BUDGET, 31)
    trapz = getattr(np, "trapezoid", None)
    if trapz is None:
        trapz = np.trapz
    area = trapz(np.interp(grid, xp, yp), grid) / AURBC_MAX_BUDGET
    required = TARGET_RECOVERY * positives
    k = int(np.searchsorted(cp, required, side="left"))
    prior_pos = cp[k - 1] if k else 0.0
    prior_mass = cw[k - 1] if k else 0.0
    budget50 = (prior_mass + (required - prior_pos) / gp[k] * gw[k]) / total
    return np.array([rec10, area, budget50])


def aggregate_plans(values, plans, methods):
    # Equal seed × generator × natural-fold weights. Each method must cover full grid.
    return np.stack([values[[i for i, plan in enumerate(plans)
                             if plan["key"][3] == method]].mean(axis=0) for method in methods])


def pair_list(methods):
    result = [(m, "IAPL", "overall") for m in methods if m != "IAPL"]
    result.extend(("RFRR-Avg", m, "mechanism") for m in
                  ("DirectPair", "ResidualNoGate", "RFRR-Beta0") if m in methods)
    if "DirectPair" in methods and "ResidualNoGate" in methods:
        result.append(("ResidualNoGate", "DirectPair", "structure"))
    if "RFRR-WG" in methods:
        result.append(("RFRR-Avg", "RFRR-WG", "objective"))
    elif "RFRR" in methods:
        result.append(("RFRR-Avg", "RFRR", "objective_historical_unmatched"))
    return result


def holm_adjust(pvalues):
    order = np.argsort(pvalues)
    sorted_p = np.array(pvalues)[order]
    adjusted = np.maximum.accumulate((len(order) - np.arange(len(order))) * sorted_p)
    out = np.empty(len(order))
    out[order] = np.minimum(1, adjusted)
    return out


def bootstrap_report(predictions, label):
    plans = prepare_plans(predictions)
    methods = list(dict.fromkeys(plan["key"][3] for plan in plans))
    require("IAPL" in methods, "Bootstrap 缺少 IAPL。")
    expected_grid = set((g, f, s) for g in DOMAINS for f in range(NATURAL_FOLDS)
                        for s in predictions.seed.unique())
    for method in methods:
        actual_grid = {p["key"][:3] for p in plans if p["key"][3] == method}
        require(actual_grid == expected_grid, f"{method} 未覆盖完整域×折×种子。")
    point_raw = np.stack([plan_metrics(p, np.ones(len(classes), dtype=int)) for p in plans])
    point_macro = aggregate_plans(point_raw, plans, methods)
    point_rows = []
    for plan, value in zip(plans, point_raw):
        point_rows.append(dict(zip(
            ["heldout_generator", "test_natural_fold", "seed", "method"], plan["key"]),
            **dict(zip(METRICS, value))))
    save_csv(label + "_point_trials.csv", pd.DataFrame(point_rows))

    # Signature includes predictions and source metadata, protecting cached bootstrap draws.
    signature = hashlib.sha256(
        pd.util.hash_pandas_object(predictions, index=False).values.tobytes()
        + json.dumps(contract, sort_keys=True).encode()
        + str(N_BOOTSTRAP).encode()
    ).hexdigest()
    checkpoint = OUT / (label + "_bootstrap_checkpoint.npz")
    draws = np.full((N_BOOTSTRAP, len(methods), len(METRICS)), np.nan)
    complete = 0
    if checkpoint.is_file():
        with np.load(checkpoint, allow_pickle=False) as saved:
            require(saved["signature"].item() == signature, "Bootstrap检查点与当前预测不一致。")
            draws = saved["draws"].copy()
            complete = int(saved["complete"].item())
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    # Generate the entire small class-count matrix for deterministic restart.
    counts = rng.multinomial(len(classes), np.full(len(classes), 1 / len(classes)),
                             size=N_BOOTSTRAP)
    for b in tqdm(range(complete, N_BOOTSTRAP), desc=label + " 共同类别Bootstrap", unit="次"):
        values = np.stack([plan_metrics(plan, counts[b]) for plan in plans])
        draws[b] = aggregate_plans(values, plans, methods)
        if (b + 1) % SAVE_EVERY == 0 or b + 1 == N_BOOTSTRAP:
            atomic_npz(checkpoint, signature=np.array(signature), draws=draws,
                       complete=np.array(b + 1), methods=np.array(methods))
    rows = []
    pairs = pair_list(methods)
    for method, reference, responsibility in pairs:
        i, j = methods.index(method), methods.index(reference)
        for k, metric in enumerate(METRICS):
            sample = draws[:, i, k] - draws[:, j, k]
            sample = sample[np.isfinite(sample)]
            point = float(point_macro[i, k] - point_macro[j, k])
            require(len(sample) >= N_BOOTSTRAP * 0.95, "有效Bootstrap不足95%，检查稀少漏检域。")
            lo, hi = np.quantile(sample, (0.025, 0.975))
            higher = metric != "required_budget_at_50"
            sign = 1 if higher else -1
            # Centered bootstrap approximate test for H0 delta=0.
            centered = sample - point
            pvalue = (1 + np.sum(sign * centered >= sign * point)) / (len(sample) + 1)
            rows.append(dict(method=method, reference=reference, responsibility=responsibility,
                             metric=metric, delta_pp=100 * point, ci_low_pp=100 * lo,
                             ci_high_pp=100 * hi, n_valid=len(sample),
                             favorable_interval=bool(lo > 0 if higher else hi < 0),
                             p_one_sided_centered_bootstrap=float(pvalue)))
    result = pd.DataFrame(rows)
    result["holm_p_recovery_mechanisms"] = np.nan
    subset = result.responsibility.eq("mechanism") & result.metric.eq("recovery_at_10")
    if subset.any():
        result.loc[subset, "holm_p_recovery_mechanisms"] = holm_adjust(
            result.loc[subset, "p_one_sided_centered_bootstrap"].to_numpy())
    save_csv(label + "_paired_intervals.csv", result)
    macro_rows = []
    for j, method in enumerate(methods):
        for k, metric in enumerate(METRICS):
            samples = draws[:, j, k]
            good = samples[np.isfinite(samples)]
            require(len(good) > 0, "宏平均区间无有效重复。")
            lo, hi = np.quantile(good, (0.025, 0.975))
            macro_rows.append(dict(method=method, metric=metric, point_pct=100 * point_macro[j, k],
                                   ci_low_pct=100 * lo, ci_high_pct=100 * hi))
    save_csv(label + "_macro_intervals.csv", pd.DataFrame(macro_rows))
    display(result.round(5))
    return result, pd.DataFrame(point_rows)


historical_intervals, historical_points = bootstrap_report(historical, "04_historical")
if MODE == "all" and BOOTSTRAP_TRAINED_RESULTS:
    trained_intervals, trained_points = bootstrap_report(trained, "05_three_seed_mean")


04_historical 共同类别Bootstrap:   0%|          | 0/2000 [00:00<?, ?次/s]

,method,reference,responsibility,metric,delta_pp,ci_low_pp,ci_high_pp,n_valid,favorable_interval,p_one_sided_centered_bootstrap,holm_p_recovery_mechanisms
0,LR-Fusion,IAPL,overall,recovery_at_10,-7.71544,-9.49707,-5.38151,2000,False,1.00000,NaN
1,LR-Fusion,IAPL,overall,aurbc_0_30,-3.82355,-4.66503,-2.93074,2000,False,1.00000,NaN
2,LR-Fusion,IAPL,overall,required_budget_at_50,1.71614,1.25137,2.11445,2000,False,1.00000,NaN
3,RFRR-Avg,IAPL,overall,recovery_at_10,1.35569,1.09949,1.74004,2000,True,0.00050,NaN
4,RFRR-Avg,IAPL,overall,aurbc_0_30,1.05901,0.98090,1.14419,2000,True,0.00050,NaN
5,RFRR-Avg,IAPL,overall,required_budget_at_50,-0.07902,-0.13491,-0.03399,2000,True,0.00300,NaN
6,RFRR,IAPL,overall,recovery_at_10,0.52542,0.30914,0.92121,2000,True,0.00250,NaN
7,RFRR,IAPL,overall,aurbc_0_30,0.38701,0.31566,0.47093,2000,True,0.00050,NaN
8,RFRR,IAPL,overall,required_budget_at_50,-0.12194,-0.17941,-0.06481,2000,True,0.00050,NaN
9,RFRR-Avg,RFRR,objective_historical_unmatched,recovery_at_10,0.83027,0.49452,1.10703,2000,True,0.00050,NaN
